# Act III — The far side: algorithms, the payoff

> *"That's the climb. You're across. Now — what was it all for?"*

Four graph algorithms, each buying you one promise from the bookend: **smarter, cheaper, more reliable.**

**Casting (decided):** the recipe graph carries exactly **one live cameo** — shortest-path, the
substitution chain, the only mechanic the domain makes legible. The money beats (PPR, HITS,
subgraph) are **judgements-solo**: pre-built artifacts toured, never live-coded. A second real
anchor — **proving-ground** — closes the *cheaper* claim with measured numbers.

**Provenance honesty (read this before the slides):**

- judgements artifacts are **SAMPLE** — illustrative shape only, names are obvious placeholders.
  They ship as samples in this pack; each cell prints its `_status` so SAMPLE vs REAL is never ambiguous.
- the proving-ground `cost_collapse` artifact is **REAL** — measured tool-use / token counts from
  committed result files (repo @ commit recorded). Verify-before-slide: the numbers on screen match
  the JSON here.

Everything below runs **offline**: no LLM, no network. The cameo replays extraction from the
committed cache; the money beats load committed JSON artifacts.

## Live cameo — shortest path *(recipe, the one thing we run live)* — *more reliable*

*"How does A relate to B, and through what?"* — the query vector search can't answer at all.
Every *"explain why"* is secretly a **shortest-path-with-constraints**.

On the recipe graph (v3, enriched with `SUBSTITUTES_FOR` edges): *no buttermilk* — what do I use,
and what's the chain that tells me so? We build the graph from the hero recipes (extraction
replayed from cache), then ask `explain_path` for the substitution route.

In [ ]:
from graphtools.data import load_hero_texts
from graphtools.extract import extract_recipe
from graphtools.graph import build_graph_v3
from graphtools.algos import explain_path

# Hero recipes -> extracted records (OFFLINE replay) -> v3 graph with SUBSTITUTES_FOR edges.
recipes = [extract_recipe(t) for _, t in load_hero_texts()]
g = build_graph_v3(recipes, with_substitutions=True)
print(f"recipe graph v3: {g.number_of_nodes()} nodes, {g.number_of_edges()} edges")

In [ ]:
# "I'm out of buttermilk." Ask the graph for the route to a substitute it knows.
path = explain_path(g, 'ingredient:buttermilk', 'ingredient:milk')
print('shortest path:', ' -> '.join(path))

# The chain isn't just nodes -- the SUBSTITUTES_FOR edge carries the 'plus' annotation
# (buttermilk -> milk *plus* acid). That annotation is the auditable 'why'.
print('\nthe chain, step by step:')
for a, b in zip(path, path[1:]):
    edge = g.get_edge_data(a, b) or {}
    sub = edge.get('SUBSTITUTES_FOR', {})
    plus = sub.get('plus')
    label = a.split(':', 1)[1]
    nxt = b.split(':', 1)[1]
    via = f'  (+ {plus})' if plus else ''
    print(f'  {label}  ->  {nxt}{via}')

print('\nReliable: no buttermilk -> use milk + an acid. Traceable, auditable, not a vibe.')

That's the **only** thing we run live. The rest of Act III is the money — toured, at scale, on a
real corpus. We switch domains: from recipes to **legal judgements**.

In [ ]:
from graphtools.bench import load_benchmark, list_benchmarks, list_sources

print('artifact sources:', list_sources())
print('judgements:     ', list_benchmarks('judgements'))
print('proving-ground: ', list_benchmarks('proving-ground'))


def banner(label, status):
    """Surface an artifact's provenance status loudly — SAMPLE vs REAL must never be ambiguous."""
    print(f"  [{label}]  {status}")

## Personalised PageRank (PPR) — *smarter*

*"Given this seed node, what's most relevant?"* Graph-native relevance — the embedding equivalent
quietly loses structure. *(Modern anchor: **HippoRAG 2, ICML 2025** — PPR as a retrieval engine.
Industry: **Pinterest Pixie** runs personalised random walks over a 3B-node / 17B-edge graph.
Verify the numbers before they hit a slide.)*

**First, the intuition — runnable, on the recipe graph.** The whole idea is one swap: plain
PageRank teleports to a *random* node (global popularity); PPR teleports back to your **seed**
(relevance *from here*). Watch the two ranks diverge on the same graph.

In [ ]:
import networkx as nx
from graphtools.algos import ppr  # nx.pagerank(personalization={seed:1.0}) on the undirected view

ug = g.to_undirected()                                   # the same v3 recipe graph as above
seed = 'ingredient:buttermilk'

glob = sorted(nx.pagerank(ug).items(), key=lambda kv: kv[1], reverse=True)   # teleport: random node
pers = ppr(g, [seed], top=6)                                                 # teleport: back to seed

print(f"GLOBAL PageRank — the popular hubs, same for every query:")
for n, _ in glob[:6]:
    print('   ', n)
print(f"\nPPR seeded on '{seed.split(':',1)[1]}' — what's related *to the seed*:")
for n, _ in pers:
    print('   ', n)
print("\nGlobal keeps surfacing the popular recipes; PPR surfaces milk / yoghurt / cream —")
print("the dairy actually wired to buttermilk. Same graph, one swap: where you teleport.")

**Now the same idea at scale**, on the real corpus — the legal judgements citation graph. PPR
seeded on one case surfaces the **landmark three hops away** that a vector search over case text
never touched, *and* hands back the citation path that proves why. *(The artifact below is a
**SAMPLE** — illustrative shape only, not a real legal finding.)*

In [ ]:
ppr = load_benchmark('ppr_landmark', source='judgements')

banner('PPR · judgements', (ppr.get('_status') or ppr.get('_provenance','')))
print()
print(f"  seed (the query case):  {ppr['seed']}")
print(f"  landmark surfaced:      {ppr['landmark']}  ({ppr['hops']} hops away)")
print('\n  citation path (the auditable why):')
for i, case in enumerate(ppr['citation_path']):
    arrow = '      ' if i == 0 else '   -> '
    print(f"{arrow}{case}")
print('\n  ^ the landmark a vector search never touched — structure carried the answer.')

## HITS / authority *(judgements, toured)* — *smarter*

*"Which nodes are authoritative hubs?"* HITS gives a split PageRank alone can't:

- **authorities** = the landmark cases everything else cites (leading authorities);
- **hubs** = survey-style judgments that cite many authorities.

Landmark-case detection, for free, from the citation graph.

In [ ]:
hits = load_benchmark('hits_landmarks', source='judgements')

banner('HITS · judgements', (hits.get('_status') or hits.get('_provenance','')))
print('\n  authorities (landmark cases — high in-citation):')
for a in hits['authorities']:
    print(f"    {a['score']:.3f}  {a['case']}")
print('\n  hubs (survey judgments — cite many authorities):')
for h in hits['hubs']:
    print(f"    {h['score']:.3f}  {h['case']}")

## Subgraph matching — *cheaper / smarter*

*"Find this shape, not this keyword."* Your query *is* a small graph (a motif); matching finds every
place it occurs. Exact = VF2 / Glasgow / Cypher; fuzzy = learned subgraph embeddings *(NeuroMatch —
foundational intuition; pair with **G-Retriever, NeurIPS 2024** / **GRAG, NAACL 2025** — pack-only,
verify before slide)*.

**First, the intuition — runnable.** The classic case SQL can't see: an **A→B→C→A money cycle**.
We draw the triangle as a pattern and let the engine find it among a tangle of accounts.
*(Industry anchors: IBM's Graph Feature Preprocessor, Elliptic2, FRAUDAR.)*

In [ ]:
import json
from pathlib import Path
from graphtools.algos import match_subgraph

# A small 'who-pays-whom' graph (the poc-fraud-ring fixture we also render on the slide).
fix = json.loads((Path('..') / 'viz' / 'fixtures' / 'poc-fraud-ring.json').read_text())
fraud = nx.DiGraph()
for e in fix['links']:
    if e['key'] == 'pays':
        fraud.add_edge(e['source'], e['target'])

# The pattern is a *shape*: three accounts in a payment cycle  a -> b -> c -> a.
pattern = nx.DiGraph([('a', 'b'), ('b', 'c'), ('c', 'a')])

rings = {frozenset(m.values()): m for m in match_subgraph(fraud, pattern)}  # dedup rotations
print(f"{fraud.number_of_nodes()} accounts, {fraud.number_of_edges()} payments — "
      f"keyword/SQL can't ask 'is there a 3-cycle?'. The shape can:")
for m in rings.values():
    cyc = [m['a'].split(':', 1)[1], m['b'].split(':', 1)[1], m['c'].split(':', 1)[1]]
    print(f"   money cycle:  {' -> '.join(cyc)} -> {cyc[0]}   (a laundering ring, invisible in rows)")

**Now the same idea at scale**, on the real corpus — a fact-pattern motif (`plea → sentence →
appeal`, sentence typed to a statutory condition) matched across the whole judgement graph, each hit
an auditable node mapping. *(The artifact below is a **SAMPLE** — illustrative shape only.)*

In [ ]:
sub = load_benchmark('subgraph_match', source='judgements')

banner('SUBGRAPH · judgements', (sub.get('_status') or sub.get('_provenance','')))
print(f"\n  pattern (the fact-pattern motif):\n    {sub['pattern']}")
print(f"\n  matches ({len(sub['matches'])}):")
for m in sub['matches']:
    print(f"    {m['case']:<42}  plea={m['plea']:<8}  appeal={m['appeal_outcome']}")

## Cheaper — the proving-ground anchor *(REAL, measured)* — *cheaper / smarter*

The *cheaper* claim, substantiated with **real** numbers from a second purpose-built repo
(`proving-ground`): an agent navigating a **code graph** vs plain grep.

The honest framing: this is **not** "finds bugs grep can't" — both arms hit on every task
(**accuracy parity**). The measured win is **efficiency that grows with codebase size**: fewer
tool-uses (and tokens) to reach the same answer. Cross-project headline: **grep 16 tool-uses ->
graph 4**.

Unlike the judgements artifacts above, these are committed measurements — verify the on-screen
numbers against the JSON.

In [ ]:
cost = load_benchmark('cost_collapse', source='proving-ground')

banner('COST · proving-ground', 'REAL measured numbers (verify-before-slide)')
prov = cost['provenance']
print(f"  repo {prov['repo']} @ {prov['commit_short']}\n")
print(f"  headline: {cost['headline']}\n")

ps = cost['summary']['powershell']
print('  PowerShell (large repo):')
print(f"    accuracy:    {ps['acc_file']}")
print(f"    grep  mean tool-uses: {ps['mean_tool_uses_grep']}")
print(f"    graph mean tool-uses: {ps['mean_tool_uses_graph']}   ({ps['tool_use_reduction']} fewer)")
print(f"    {ps['cross_project_headline']}")

In [ ]:
from pathlib import Path
from IPython.display import Image

# The cost-collapse figure (REAL): accuracy parity, navigation cost collapses as the repo grows.
png = Path('..') / 'artifacts' / 'proving-ground' / 'cost_collapse.png'
Image(filename=str(png))

## Close — the bookend

> *"You crossed the valley. Here's what the toolkit bought you."*

You walked the pipeline left to right — **Extract → Curate → Project → Retrieve** — and on the far
side the algorithms paid it back:

- **PPR** surfaced the landmark three hops away that vector search missed — *smarter*.
- **HITS** named the authorities and hubs the corpus already knew — *smarter*.
- **Subgraph matching** found every case fitting the fact pattern at scale — *cheaper / smarter*.
- **Shortest path** gave an auditable substitution chain — the *"explain why"* — *more reliable*.
- And the **code-graph cost collapse** (16 -> 4 tool-uses, accuracy held) — *cheaper*, for real.

Done right, graph-native structure makes your AI apps **smarter, cheaper, and more reliable.**
That's the plateau. That's what it was all for.

*(Note: the judgements numbers in this pack are **SAMPLE**; the proving-ground numbers are
measured. Don't ship a placeholder as a fact.)*